# 01 - Data extraction

Downloads the CRSP daily returns of the sector ETFs, builds the daily stress events and downloads the VIX.

Author: Luca Persia (2026), Università della Svizzera italiana, Lugano, Switzerland

In [ ]:
%pip install wrds yfinance

In [ ]:
from pathlib import Path


def find_repo_root(start: Path = Path.cwd()) -> Path:
    """Return the repository root, i.e. the first folder above `start` containing both data/ and src/."""
    for p in [start, *start.parents]:
        if (p / "data").is_dir() and (p / "src").is_dir():
            return p
    raise FileNotFoundError(f"Repository root not found above {start}")


ROOT_DIR    = find_repo_root()
RAW_DIR     = ROOT_DIR / "data" / "raw"
EVENTS_DIR  = ROOT_DIR / "data" / "events"
RESULTS_DIR = ROOT_DIR / "data" / "results"
for d in (RAW_DIR, EVENTS_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

In [ ]:
import pandas as pd
import wrds
import yfinance as yf

# Connection to WRDS (asks for username and password the first time).
db = wrds.Connection()

In [ ]:
# SETTINGS

# Select Sector SPDR ETFs of the paper.
tickers = ["XLB", "XLE", "XLF", "XLI", "XLK", "XLU", "XLV", "XLY"]
ticker_sql = "(" + ",".join(f"'{x}'" for x in tickers) + ")"

# Extended samples on which the thresholds are computed; the observation windows are applied later in MATLAB.
crisis_windows = {
    "GFC_2008":   ("2006-01-01", "2012-01-01"),
    "COVID_2020": ("2019-01-01", "2024-01-01"),
}

# Tail quantile: Q = 0.95 means the worst 5% of daily returns of each sector.
Q = 0.95
min_obs = 20

In [ ]:
# STRESS EVENTS

for crisis_name, (start_date, end_date) in crisis_windows.items():

    # Daily CRSP data (simple returns).
    sql_query = f"""
        SELECT b.ticker, a.date, ABS(a.prc) AS price, a.ret AS return, a.vol AS volume
        FROM crsp.dsf AS a
        JOIN crsp.stocknames AS b ON a.permno = b.permno
        WHERE b.ticker IN {ticker_sql}
          AND a.date >= '{start_date}' AND a.date <= '{end_date}'
          AND a.date BETWEEN b.namedt AND b.nameenddt
        ORDER BY b.ticker, a.date
    """
    sector_data = db.raw_sql(sql_query, date_cols=["date"])
    sector_data["ticker"] = sector_data["ticker"].astype(str).str.strip()
    for col in ("price", "return", "volume"):
        sector_data[col] = pd.to_numeric(sector_data[col], errors="coerce")
    sector_data = (sector_data.dropna(subset=["ticker", "date", "return"])
                              .sort_values(["ticker", "date"])
                              .reset_index(drop=True))
    sector_data.to_csv(RAW_DIR / f"daily_prices_{crisis_name}.csv", index=False)

    # Left-tail exceedances with a static, sector-specific threshold over the whole sample.
    events = []
    for ticker, g in sector_data.groupby("ticker"):
        if g["return"].count() < min_obs:
            continue
        threshold = g["return"].quantile(1 - Q)
        hits = g[g["return"] <= threshold]
        for _, row in hits.iterrows():
            events.append({"date": row["date"], "market": ticker, "price": row["price"],
                           "return": row["return"], "threshold": threshold,
                           "exceedance": threshold - row["return"]})
    df_events = pd.DataFrame(events)

    # One event per day: we keep the sector with the largest exceedance (ties broken by ticker).
    df_events = (df_events
                 .sort_values(["date", "exceedance", "market"], ascending=[True, False, True])
                 .drop_duplicates(subset=["date"], keep="first")
                 .sort_values("date")
                 .reset_index(drop=True))

    # Raw mark: exceedance scaled by its sample mean (the log transformation of Eq. 4 is applied in MATLAB).
    origin = df_events["date"].min()
    df_events["t_days"] = (df_events["date"] - origin).dt.days.astype(float)
    df_events["type"] = df_events["market"].map({mkt: i + 1 for i, mkt in enumerate(sorted(df_events["market"].unique()))})
    df_events["mark"] = df_events["exceedance"] / df_events["exceedance"].mean()

    out = df_events[["date", "market", "t_days", "type", "price", "return", "threshold", "mark"]].copy()
    out["date"] = out["date"].dt.strftime("%Y-%m-%d")
    out.to_csv(EVENTS_DIR / f"events_daily_{crisis_name}_q{int(Q * 100)}_oneperday.csv", index=False)
    print(f"{crisis_name}: {len(out)} events")

In [ ]:
# VIX FROM YAHOO FINANCE

iv_data = yf.download(["^VIX", "^SKEW"], start="2006-01-01", end="2024-01-01")["Close"].reset_index()
iv_data = iv_data.rename(columns={"Date": "date", "^VIX": "VIX", "^SKEW": "SKEW"})
iv_data["date"] = pd.to_datetime(iv_data["date"]).dt.strftime("%Y-%m-%d")
iv_data = iv_data.dropna(subset=["VIX", "SKEW"], how="all")
iv_data.to_csv(RAW_DIR / "options_signals.csv", index=False)